# 03 — Model Training & Evaluation

This notebook explains the online machine learning approach, how the base model is trained,
how we evaluate it, and how to interpret the results.

## 1. Why Online Learning?

Traditional ML trains once on a fixed dataset, then predicts. This has problems for train delays:

- **Timetables change** — new lines, schedule revisions, seasonal patterns
- **Concept drift** — summer delays differ from winter delays
- **You don't have a dataset on day 1** — you only have live data as it arrives

**Online learning** (also called incremental learning) solves this:
- The model updates with every new confirmed arrival — no batch retraining
- It adapts automatically to new patterns over time
- Works from the very first data point

```
Traditional ML:   [large dataset] -> train -> fixed model -> predict forever
Online ML:        predict -> observe truth -> learn -> predict -> observe -> learn -> ...
```

## 2. The HoeffdingAdaptiveTreeRegressor

We use River's `HoeffdingAdaptiveTreeRegressor` (HATR). It's a decision tree that:

- **Grows incrementally** — adds new splits only when there's enough statistical evidence
  (uses the Hoeffding bound to decide when a split is confident enough)
- **Adapts to drift** — uses ADWIN (Adaptive Windowing) to detect when a subtree's accuracy
  degrades and resets that branch
- **Scales in O(1)** — each `learn_one` call takes constant time, regardless of how many
  trips have been learned

| Property | Value |
|---|---|
| Time per learn_one | O(1) |
| Memory | O(tree size) — grows slowly |
| GPU needed | No |
| Retraining needed | Never (unless you want a fresh start) |

In [ ]:
import sys
sys.path.append('..')

from river import tree, metrics
from preprocessing import build_features, compute_delay_min

# Create a fresh model
model = tree.HoeffdingAdaptiveTreeRegressor()
print(model)

## 3. Base Training from CSV

Before the live pipeline starts, we do a one-time "warm-up" training on historical CSV data.
This gives the model a starting point so it doesn't predict random values on day 1.

The `train_base.py` script:
1. Loads a CSV export of historical trips
2. Filters out IC/ICE
3. Sorts by `scheduled_arr` (chronologically)
4. Trains on everything except the last 3 days (held out for evaluation)
5. Evaluates and saves `model.pkl` + `model_eval.json`

In [ ]:
# Simulate training on a small synthetic dataset to illustrate the loop
import random
random.seed(42)

from datetime import datetime, timedelta

def make_synthetic_trip(i):
    hour = random.randint(6, 22)
    base = datetime(2026, 5, 1, hour, random.randint(0, 59))
    true_delay = max(0, random.gauss(2, 4))  # average 2 min delay
    return {
        'scheduled_arr': base,
        'actual_arr':    base + timedelta(minutes=true_delay),
        'scheduled_dep': base - timedelta(minutes=5),
        'actual_dep':    (base - timedelta(minutes=5)) + timedelta(minutes=true_delay * 0.8),
        'train_type': random.choice(['S', 'RB', 'RE']),
        'line': random.choice(['S1', 'S2', 'S3', 'S4']),
        'station_name': 'Stuttgart Hbf',
        'station_eva': '8000096',
        'direction': 'Plochingen',
        'cancelled': False,
        'ppth': 'Kirchheim (T)|Plochingen|Esslingen|Stuttgart Hbf|Zuffenhausen',
        'upstream_delay_min': None,
    }

mae_metric = metrics.MAE()
synthetic_model = tree.HoeffdingAdaptiveTreeRegressor()

for i in range(200):
    row = make_synthetic_trip(i)
    features = build_features(row)
    delay = compute_delay_min(row['scheduled_arr'], row['actual_arr'])

    if delay is not None:
        pred = synthetic_model.predict_one(features)
        mae_metric.update(delay, pred)
        synthetic_model.learn_one(features, delay)

print(f'Trained on 200 synthetic trips')
print(f'MAE on training stream: {mae_metric.get():.2f} min')

## 4. Evaluation Metrics Explained

| Metric | Formula | Good value | What it tells you |
|---|---|---|---|
| **MAE** | mean(|actual - predicted|) | < 2 min | Average error size |
| **RMSE** | sqrt(mean((actual - predicted)^2)) | < 4 min | Penalizes large errors more |
| **R²** | 1 - SS_res/SS_tot | > 0.5 | How much better than predicting the mean |
| **Within ±2 min** | % of trips with error <= 2 min | > 60% | Practical accuracy |
| **Within ±5 min** | % of trips with error <= 5 min | > 85% | Broad accuracy |

**R² interpretation:**
- R² = 1.0 → perfect predictions
- R² = 0.0 → model is no better than always predicting the mean delay
- R² < 0 → model is worse than predicting the mean (baseline is bad)

A negative baseline R² means the training CSV data was old enough that its patterns
no longer matched the live network — the online learner quickly compensates.

In [ ]:
import json
import os

eval_path = '../model_eval.json'
if os.path.exists(eval_path):
    with open(eval_path) as f:
        ev = json.load(f)
    print('Baseline model evaluation (from train_base.py):')
    print(f"  Trained on  : {ev['trained_trips']:,} trips")
    print(f"  Tested on   : {ev['test_trips']:,} trips")
    print(f"  MAE         : {ev['mae_min']:.3f} min")
    print(f"  RMSE        : {ev['rmse_min']:.3f} min")
    print(f"  R2          : {ev['r2']:.4f}")
    print(f"  Within +/-2 : {ev['within_2min_pct']:.1f}%")
    print(f"  Within +/-5 : {ev['within_5min_pct']:.1f}%")
    print(f"  CSV source  : {ev['csv_source']}")
else:
    print('model_eval.json not found — run train_base.py first.')

## 5. The Learning Loop in pipeline.py

Every 60 seconds, for every trip with a confirmed actual arrival:

```python
features = build_features(row)         # 29 features
pred     = model.predict_one(features) # current prediction
delay    = actual_arr - sched_arr      # ground truth

model.learn_one(features, delay)       # update the tree
```

**Key guards:**
- `learned_keys` set — prevents re-learning the same trip arrival if seen across multiple cycles
- Delay capped to `[-120, +120]` min — ignores extreme outliers that would corrupt the model
- Prediction capped to `[-30, +120]` min — prevents absurd outputs before the tree is well-trained
- Model saved every 500 trips (atomic overwrite via temp file)

## 6. Retraining Strategy

After collecting a few weeks of live data:

1. Export from PostgreSQL:
   ```sql
   COPY (SELECT * FROM trips ORDER BY scheduled_arr)
   TO '/path/trips_final.csv' CSV HEADER;
   ```

2. Run `train_base.py` again with the live CSV

3. The new `model.pkl` will have:
   - Seen route topology features (`ppth`, `route_position_pct`) from day one
   - Seen upstream delay features in training
   - Been trained on current timetable patterns (not old CSV data)

Expected improvement after retraining on ~2 weeks of live data:
- MAE: 2.5 → 1.5–2.0 min
- Within ±2 min: 65% → 70–75%
- R²: 0.4 → 0.6+

## 7. Model Improvement Tracking

Every 500 trips learned, the pipeline appends to `model_log.jsonl`:

```json
{"ts": "2026-05-30T08:00:00", "trips_learned": 500, "mae_100": 2.31, "mae_500": 2.58}
{"ts": "2026-05-30T14:00:00", "trips_learned": 1000, "mae_100": 2.15, "mae_500": 2.40}
```

- `mae_100` — rolling MAE over the last 100 trips (noisy but responsive)
- `mae_500` — rolling MAE over the last 500 trips (smooth, shows real trend)

The dashboard shows these as a line chart under **Model Improvement over Time**.

In [ ]:
import json
import os

log_path = '../model_log.jsonl'
if os.path.exists(log_path):
    rows = []
    with open(log_path) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    print(f'Model log: {len(rows)} checkpoints')
    for row in rows:
        print(f"  {row['ts']}  trips={row['trips_learned']:,}  "
              f"mae_100={row['mae_100']:.2f}  mae_500={row['mae_500']:.2f}")
else:
    print('model_log.jsonl not yet created — appears after first 500 trips learned.')